<a href="https://colab.research.google.com/github/ThLanBot42/DHU_AI_THL_CV_COMP_COMPETITION/blob/main/firegazing_V3.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, shutil, random
from pathlib import Path
from google.colab import drive

# ========== 1. 挂载 Drive ==========
drive.mount('/content/drive', force_remount=True)

# ========== 2. 路径常量（按你实际修改）==========
DRIVE_ROOT = Path('/content/drive/MyDrive/AI_competition')
LOCAL_ROOT = Path('/content/datasets/wheat')
DRIVE_PSEUDO = DRIVE_ROOT / 'pseudo_labels_train' / 'labels'

# ========== 3. 恢复被拆散的 val 图片到 train ===========
train_img = LOCAL_ROOT / 'images/train'
val_img   = LOCAL_ROOT / 'images/val'
train_img.mkdir(parents=True, exist_ok=True)
val_img.mkdir(parents=True, exist_ok=True)

moved = 0
for f in val_img.iterdir():
    if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}:
        dst = train_img / f.name
        if dst.exists():
            shutil.move(str(f), str(dst))
            moved += 1
print(f"✅ 已从 val 恢复 {moved} 张图片到 images/train")

# ========== 4. 建立本地高速目录结构 ==========
for split in ['train','val','test']:
    (LOCAL_ROOT/'images'/split).mkdir(parents=True, exist_ok=True)
    if split != 'test':
        (LOCAL_ROOT/'labels'/split).mkdir(parents=True, exist_ok=True)

# ========== 5. 符号链接图片到本地（不复制，秒完成）==========
for split in ['train','test']:
    d_src = DRIVE_ROOT / 'images' / split
    d_dst = LOCAL_ROOT / 'images' / split
    if d_src.exists():
        for f in d_src.iterdir():
            if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}:
                link = d_dst / f.name
                if not link.exists():
                    os.symlink(str(f), str(link))

n_train = len([f for f in (LOCAL_ROOT/'images/train').iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}])
n_test  = len([f for f in (LOCAL_ROOT/'images/test').iterdir()  if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}])
print(f"✅ 本地图片链接 | train: {n_train} | test: {n_test}")

# ========== 6. 搬运伪标签 + 强制修复类别 ===========
local_lbl = LOCAL_ROOT / 'labels/train'
local_lbl.mkdir(parents=True, exist_ok=True)

if not DRIVE_PSEUDO.exists():
    raise FileNotFoundError(f"❌ 找不到伪标签目录: {DRIVE_PSEUDO}\n请确认 Drive 里已重命名为 pseudo_labels_train")

pseudo_files = [f for f in DRIVE_PSEUDO.iterdir() if f.suffix=='.txt']
for f in pseudo_files:
    shutil.copy(str(f), str(local_lbl/f.name))
print(f"✅ 伪标签已搬运: {len(pseudo_files)} 个")

# 修复：强制索引为 0，清洗非法坐标
fixed = 0
for fpath in local_lbl.iterdir():
    if fpath.suffix != '.txt':
        continue
    with open(fpath) as fp:
        lines = fp.readlines()
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            parts[0] = '0'
            try:
                x, y, bw, bh = map(float, parts[1:5])
                if 0 <= x <= 1 and 0 <= y <= 1 and 0 < bw <= 1 and 0 < bh <= 1:
                    new_lines.append(' '.join(parts) + '\n')
            except:
                continue
    with open(fpath, 'w') as fp:
        fp.writelines(new_lines)
    if new_lines:
        fixed += 1
print(f"✅ 已修复/清洗: {fixed} 个标签文件")

# ========== 7. 划分验证集（10%，Copy 模式）==========
random.seed(42)
lbl_names = {f.stem for f in local_lbl.iterdir() if f.suffix=='.txt'}
img_files = [f for f in train_img.iterdir()
             if f.stem in lbl_names
             and f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]

random.shuffle(img_files)
val_num = max(1, int(len(img_files) * 0.1))
val_imgs = img_files[:val_num]

for img_path in val_imgs:
    shutil.copy(str(img_path), str(val_img/img_path.name))
    lbl = local_lbl / (img_path.stem + '.txt')
    if lbl.exists():
        shutil.copy(str(lbl), str(LOCAL_ROOT/'labels/val'/lbl.name))

print(f"✅ 验证集: {val_num} 张 | 训练集剩余: {len(img_files)-val_num} 张")

# ========== 8. 写出官方格式的 data.yaml ===========
yaml_content = f"""path: {LOCAL_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: wheat_head
"""
with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content.strip() + '\n')
print("✅ data.yaml 已写入 /content/data.yaml")

# ========== 9. 数据完整性检查 ===========
for split in ['train','val']:
    idir = LOCAL_ROOT / 'images' / split
    ldir = LOCAL_ROOT / 'labels' / split
    imgs = {f.stem for f in idir.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}}
    lbls = {f.stem for f in ldir.iterdir() if f.suffix=='.txt'}
    matched = imgs & lbls
    print(f"[{split}] 图片: {len(imgs)} | 标签: {len(lbls)} | 匹配: {len(matched)}")
    if imgs - lbls:
        print(f"  ⚠️ {len(imgs-lbls)} 张图片无标签（模型未检测到麦穗，属正常）")
    if lbls - imgs:
        print(f"  ❌ {len(lbls-imgs)} 个标签无对应图片（异常）")

assert any(f.suffix=='.txt' for f in local_lbl.iterdir()), "❌ train 无标签，检查伪标签路径！"
print("\n🎯 数据准备完毕，请继续运行 train.py / test.py / submit_infer.py")

Mounted at /content/drive
✅ 已从 val 恢复 120 张图片到 images/train
✅ 本地图片链接 | train: 4116 | test: 1025
✅ 伪标签已搬运: 1208 个
✅ 已修复/清洗: 1208 个标签文件
✅ 验证集: 120 张 | 训练集剩余: 1088 张
✅ data.yaml 已写入 /content/data.yaml
[train] 图片: 4116 | 标签: 1208 | 匹配: 1208
  ⚠️ 2908 张图片无标签（模型未检测到麦穗，属正常）
[val] 图片: 120 | 标签: 120 | 匹配: 120

🎯 数据准备完毕，请继续运行 train.py / test.py / submit_infer.py


In [4]:
!pip install -q ultralytics
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.5 MB/s eta 0:00:00
PyTorch 版本: 2.10.0+cu128
CUDA 可用: True
GPU: NVIDIA A100-SXM4-80GB


In [5]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="data.yaml",
    imgsz=1024,
    epochs=50,
    batch=32,
    device=0,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    close_mosaic=10,
    # 以下保存到 Drive，防 Colab 断连丢失权重（强烈推荐）
    project="/content/drive/MyDrive/AI_competition/training_runs",
    name="wheat_v8m_1024",
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b66abb240e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
from pathlib import Path

# 检查 Drive 里的权重路径
drive_weight = Path("/content/drive/MyDrive/AI_competition/training_runs/wheat_v8m_1024/weights/best.pt")
local_weight = Path("runs/detect/wheat_v8m_1024/weights/best.pt")

print(f"Drive 权重存在: {drive_weight.exists()} | 路径: {drive_weight}")
print(f"本地权重存在: {local_weight.exists()} | 路径: {local_weight}")

# 如果 Drive 里有，直接用它
if drive_weight.exists():
    print("✅ 训练已完成，权重在 Drive 里")
else:
    print("❌ 权重不存在，你需要先运行 train.py 完成训练")

Drive 权重存在: True | 路径: /content/drive/MyDrive/AI_competition/training_runs/wheat_v8m_1024/weights/best.pt
本地权重存在: False | 路径: runs/detect/wheat_v8m_1024/weights/best.pt
✅ 训练已完成，权重在 Drive 里


In [8]:
from pathlib import Path
from ultralytics import YOLO
import csv

# ==================== 自动检测权重路径（Drive 优先）====================
drive_pt = Path("/content/drive/MyDrive/AI_competition/training_runs/wheat_v8m_1024/weights/best.pt")
local_pt = Path("runs/detect/wheat_v8m_1024/weights/best.pt")

if drive_pt.exists():
    WEIGHTS = str(drive_pt)
    print("✅ 使用 Drive 权重")
elif local_pt.exists():
    WEIGHTS = str(local_pt)
    print("✅ 使用本地权重")
else:
    raise FileNotFoundError("❌ 找不到 best.pt，请先运行 train.py")

# ==================== 推理配置 ====================
IMAGES_DIR = Path("/content/datasets/wheat/images/test")
OUT_CSV = Path("/content/drive/MyDrive/AI_competition/submission.csv")
IMGSZ = 1024
CONF = 0.001
DEVICE = "0"

# ==================== 执行推理并生成 CSV ====================
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
paths = sorted([x for x in IMAGES_DIR.iterdir() if x.suffix.lower() in exts])

if not paths:
    raise SystemExit(f"❌ 测试集目录为空: {IMAGES_DIR}")

model = YOLO(WEIGHTS)
rows = []

for img_path in paths:
    image_id = img_path.stem
    r = model.predict(
        source=str(img_path),
        imgsz=IMGSZ,
        conf=CONF,
        device=DEVICE,
        verbose=False,
    )[0]

    if r.boxes is None or len(r.boxes) == 0:
        continue

    xywhn = r.boxes.xywhn.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()
    clss = r.boxes.cls.cpu().numpy().astype(int)

    for i in range(len(xywhn)):
        xc, yc, bw, bh = map(float, xywhn[i])
        rows.append({
            "image_id": image_id,
            "class_id": int(clss[i]),
            "x_center": round(xc, 6),
            "y_center": round(yc, 6),
            "width": round(bw, 6),
            "height": round(bh, 6),
            "confidence": round(float(confs[i]), 6),
        })

# 写入 CSV
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUT_CSV.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=[
        "image_id", "class_id", "x_center", "y_center",
        "width", "height", "confidence"
    ])
    w.writeheader()
    w.writerows(rows)

print(f"✅ 提交文件已生成: {OUT_CSV}")
print(f"📊 共 {len(rows)} 个检测框，覆盖 {len(set(r['image_id'] for r in rows))} 张图片")

✅ 使用 Drive 权重
✅ 提交文件已生成: /content/drive/MyDrive/AI_competition/submission.csv
📊 共 21222 个检测框，覆盖 968 张图片


In [1]:
# ========== 断联后一键恢复：五个文件重建 + 打包 ==========
#此处发现提交格式
!pip install -q ultralytics
from google.colab import drive; drive.mount('/content/drive', force_remount=True)

import shutil
from pathlib import Path
from datetime import datetime

base = Path('/content/drive/MyDrive/AI_competition')
base.mkdir(parents=True, exist_ok=True)

# 1. 写入四个代码文件到 Drive（永久保存）
files = {
    'data.yaml': """path: /content/datasets/wheat
train: images/train
val: images/val
test: images/test

names:
  0: wheat_head
""",
    'train.py': """from ultralytics import YOLO
model = YOLO("yolov8m.pt")
model.train(
    data="data.yaml", imgsz=1024, epochs=50, batch=32, device=0,
    optimizer="SGD", lr0=0.01, lrf=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3.0, close_mosaic=10,
    augment=True, mosaic=1.0, scale=0.5, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    project="/content/drive/MyDrive/AI_competition/training_runs",
    name="wheat_v8m_1024", exist_ok=True,
)
""",
    'test.py': """from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/AI_competition/training_runs/wheat_v8m_1024/weights/best.pt")
model.val(data="data.yaml", imgsz=1024, batch=32, device=0, split="test")
""",
    'submit_infer.py': """import csv
from pathlib import Path
from ultralytics import YOLO

IMAGES_DIR = Path("/content/datasets/wheat/images/test")
WEIGHTS = Path("/content/drive/MyDrive/AI_competition/training_runs/wheat_v8m_1024/weights/best.pt")
OUT_CSV = Path("/content/drive/MyDrive/AI_competition/submission.csv")
IMGSZ, CONF, DEVICE = 1024, 0.001, "0"

def main():
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    paths = sorted([x for x in IMAGES_DIR.iterdir() if x.suffix.lower() in exts])
    if not paths: raise SystemExit(f"no images in {IMAGES_DIR}")
    model = YOLO(str(WEIGHTS))
    rows = []
    for img_path in paths:
        image_id = img_path.stem
        r = model.predict(source=str(img_path), imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]
        if r.boxes is None or len(r.boxes)==0: continue
        xywhn = r.boxes.xywhn.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy().astype(int)
        for i in range(len(xywhn)):
            xc,yc,bw,bh = map(float, xywhn[i])
            rows.append({"image_id":image_id,"class_id":int(clss[i]),"x_center":xc,"y_center":yc,"width":bw,"height":bh,"confidence":float(confs[i])})
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with OUT_CSV.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["image_id","class_id","x_center","y_center","width","height","confidence"])
        w.writeheader(); w.writerows(rows)
    print(f"wrote {len(rows)} boxes to {OUT_CSV}")
if __name__=="__main__": main()
"""
}

for name, content in files.items():
    with open(base / name, 'w') as f:
        f.write(content)
    print(f"✅ 已写入 Drive: {name}")

# 2. 检查 submission.csv 是否存在
csv_path = base / 'submission.csv'
if csv_path.exists():
    print(f"✅ submission.csv 存在 ({csv_path.stat().st_size/1024:.1f} KB)")
else:
    print("⚠️ submission.csv 缺失！需要运行上面的 submit_infer.py 生成")

# 3. 把五个文件打包成 zip
zip_path = base / 'submit_package.zip'
with shutil.ZipFile(zip_path, 'w') as zf:
    for name in ['data.yaml', 'train.py', 'test.py', 'submit_infer.py', 'submission.csv']:
        fpath = base / name
        if fpath.exists():
            zf.write(fpath, arcname=name)
        else:
            print(f"❌ 打包失败：缺少 {name}")

print(f"\n📦 打包完成: {zip_path}")
print("🎯 去 Drive 下载这个 zip，里面就是五个文件，直接提交")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00
Mounted at /content/drive
✅ 已写入 Drive: data.yaml
✅ 已写入 Drive: train.py
✅ 已写入 Drive: test.py
✅ 已写入 Drive: submit_infer.py
✅ submission.csv 存在 (2330.5 KB)


AttributeError: module 'shutil' has no attribute 'ZipFile'

In [2]:
import zipfile
from pathlib import Path

base = Path('/content/drive/MyDrive/AI_competition')
zip_path = base / 'submit_package.zip'

with zipfile.ZipFile(zip_path, 'w') as zf:
    for name in ['data.yaml', 'train.py', 'test.py', 'submit_infer.py', 'submission.csv']:
        fpath = base / name
        if fpath.exists():
            zf.write(fpath, arcname=name)
            print(f"✅ 已打包: {name}")
        else:
            print(f"❌ 缺少: {name}")

print(f"\n📦 打包完成: {zip_path}")

✅ 已打包: data.yaml
✅ 已打包: train.py
✅ 已打包: test.py
✅ 已打包: submit_infer.py
✅ 已打包: submission.csv

📦 打包完成: /content/drive/MyDrive/AI_competition/submit_package.zip
